## Import Libraries

In [1]:
import os
import time
import pandas as pd
import requests
from tqdm import tqdm
from urllib.parse import quote_plus
from bs4 import BeautifulSoup
import re

## Define Parameters and Keywords

In [ ]:
# Replace with your keys (or load via .env)
CLIENT_ID = "xxx"
CLIENT_SECRET = "xxx"

HEADERS = {
    "X-Naver-Client-Id": CLIENT_ID,
    "X-Naver-Client-Secret": CLIENT_SECRET
}

KEYWORDS = ["경제", "물가", "인플레이션", "금리", "부동산", "경기침체"]
def build_query(base_kw):
    return f"2025년 9월 {base_kw}"
SAVE_PATH = "data/economic_news_api.csv"
os.makedirs("data", exist_ok=True)

## Define Fetch Function

In [3]:
def fetch_news(query, display=100, start=1, sort="date"):
    """Fetch news from Naver OpenAPI"""
    base_url = "https://openapi.naver.com/v1/search/news.json"
    params = {
        "query": query,
        "display": display,
        "start": start,
        "sort": sort
    }
    res = requests.get(base_url, headers=HEADERS, params=params)
    if res.status_code == 200:
        return res.json().get("items", [])
    else:
        print(f"Error {res.status_code}: {res.text}")
        return []

## Crawl Multiple Keywords

In [4]:
def get_article_content(url):
    try:
        res = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
        res.raise_for_status()
        soup = BeautifulSoup(res.text, 'html.parser')
        content = soup.select_one('#dic_area') or soup.select_one('.newsct_article')
        if content:
            text = content.get_text(separator=' ', strip=True)
            text = re.sub(r'\[.*?\]|\(.*?\)|사진.*?기자', '', text)
            return text
        return None
    except:
        return None

all_results = []

for kw in tqdm(KEYWORDS, desc="Keywords (9월)"):
    query = build_query(kw)
    for start in range(1, 100, 100):
        items = fetch_news(query, display=100, start=start)
        if not items:
            break
        for it in items:
            article_text = get_article_content(it["link"])
            all_results.append({
                "keyword": kw,
                "title": it["title"].replace("<b>", "").replace("</b>", ""),
                "summary": it["description"].replace("<b>", "").replace("</b>", ""),
                "link": it["link"],
                "originallink": it.get("originallink", ""),
                "pubDate": it["pubDate"],
                "content": article_text
            })
        time.sleep(0.5)

Keywords (9월): 100%|████████████████████████████████████████████████████████████████████| 6/6 [02:46<00:00, 27.83s/it]


## Save Results

In [5]:
df = pd.DataFrame(all_results)
df.drop_duplicates(subset=["link"], inplace=True)
df["pubDate"] = pd.to_datetime(df["pubDate"])
df.to_csv("data/economic_news_sep2025.csv", index=False, encoding="utf-8-sig")

print(f"Saved {len(df)} rows (September-focused data)")
df.head()

Saved 463 rows (September-focused data)


,keyword,title,summary,link,originallink,pubDate,content
0,경제,발전공기업 태양광·풍력 보급계획 가속화 필요…2040년 탈석탄 대비해...,11일 한국전력 경영연구원이 최근 발간한 '2024년 글로벌 재생에너지 발전용량 및...,https://daily.hankooki.com/news/articleView.ht...,https://daily.hankooki.com/news/articleView.ht...,2025-10-11 15:00:00+09:00,None
1,경제,9월 글로벌 선박 수주량 44% 급감…K-조선 39% '세계 2위',/연합뉴스 | 한스경제=이성노 기자 | 지난달 글로벌 선박 수주량이 전년 동기 대비...,http://www.hansbiz.co.kr/news/articleView.html...,http://www.hansbiz.co.kr/news/articleView.html...,2025-10-11 14:12:00+09:00,None
2,경제,"IMF, 14일 세계경제전망 발표...국내 고용시장 상황 통계치 공개도",/ 뉴시스 국제통화기금(IMF)이 다음주에 올해 마지막 세계경제전망을 발표한다. 지...,http://www.metroseoul.co.kr/article/2025101150...,http://www.metroseoul.co.kr/article/2025101150...,2025-10-11 14:00:00+09:00,None
3,경제,"서울 6억원 미만 아파트, 10년 새 5분의 1로…&quot;'내 집 마련' 발판 ...",11일 부동산 중개업체 집토스에 따르면 2015년에서 올해 9월 현재까지 신고된 서...,https://www.ajunews.com/view/20251002112458992,https://www.ajunews.com/view/20251002112458992,2025-10-11 14:00:00+09:00,None
4,경제,"순천시 우수봉사자, 제천·영월서 자원봉사 열정 재충전",아주경제=순천=박기현 기자 qkrqkr@hanmail.net 2025년 우수자원봉사...,https://www.ajunews.com/view/20251011120814416,https://www.ajunews.com/view/20251011120814416,2025-10-11 12:28:00+09:00,None
